# 🛒 Sales Revenue Forecasting — ML Project
## Retail Sales Prediction using Machine Learning

> **Goal**: Predict net revenue for retail transactions using historical sales data across regions, categories, and channels.

---

### 📋 Project Summary
| Item | Details |
|------|--------|
| **Dataset** | 3,000 synthetic retail transactions (2021–2024) |
| **Target** | `net_revenue` (₹ after discount & returns) |
| **Best Model** | Gradient Boosting Regressor |
| **R² Score** | **0.9963** |
| **Tech Stack** | Python · Pandas · Scikit-learn · Matplotlib · Seaborn |

## 📦 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings, joblib
warnings.filterwarnings('ignore')

print('Libraries loaded ✅')
print(f'Pandas: {pd.__version__} | NumPy: {np.__version__}')

## 📂 2. Load & Explore Dataset

In [ ]:
df = pd.read_csv('../data/sales_data.csv', parse_dates=['date'])
print(f'Shape: {df.shape}')
print(f'Date range: {df["date"].min().date()} → {df["date"].max().date()}')
df.head()

In [ ]:
# Data types and missing values
print('=== Dataset Info ===')
print(df.dtypes)
print(f'\nMissing values:\n{df.isnull().sum()}')

In [ ]:
# Statistical summary of numeric columns
df.describe().round(2)

In [ ]:
# Unique values in categorical columns
for col in ['region','category','channel']:
    print(f'{col}: {df[col].unique().tolist()}')

## 📊 3. Exploratory Data Analysis (EDA)

### 3.1 Monthly Revenue Trend

In [ ]:
monthly = df.groupby(['year','month'])['revenue'].sum().reset_index()
monthly['period'] = pd.to_datetime(monthly[['year','month']].assign(day=1))
monthly = monthly.sort_values('period')

fig, ax = plt.subplots(figsize=(14,5), facecolor='#0D1117')
ax.set_facecolor('#0D1117')
ax.fill_between(monthly['period'], monthly['revenue'], alpha=0.15, color='#58A6FF')
ax.plot(monthly['period'], monthly['revenue'], color='#58A6FF', lw=2.5, marker='o', ms=4)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x/1e5:.0f}L'))
ax.set_title('Monthly Revenue Trend (2021–2024)', fontsize=15, fontweight='bold', color='#E6EDF3')
ax.tick_params(colors='#E6EDF3'); ax.spines[['top','right']].set_visible(False)
ax.grid(True, axis='y', color='#21262D')
plt.tight_layout(); plt.show()
print(f'Peak month revenue: ₹{monthly["revenue"].max():,.0f}')

### 3.2 Revenue by Category & Channel

In [ ]:
cat_chan = df.groupby(['category','channel'])['revenue'].sum().unstack(fill_value=0)
print('Revenue breakdown (₹ Lakhs):')
(cat_chan/1e5).round(1)

In [ ]:
# Top performing categories
cat_rev = df.groupby('category').agg(
    total_revenue=('revenue','sum'),
    total_profit=('profit','sum'),
    avg_units=('units_sold','mean'),
    transactions=('revenue','count')
).sort_values('total_revenue', ascending=False)
cat_rev['profit_pct'] = (cat_rev['total_profit']/cat_rev['total_revenue']*100).round(1)
cat_rev.round(0)

### 3.3 Seasonal Patterns

In [ ]:
monthly_avg = df.groupby('month')['revenue'].mean()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
print('Average revenue by month:')
for m, name in enumerate(month_names, 1):
    bar = '█' * int(monthly_avg[m]/monthly_avg.max()*30)
    print(f'{name}: {bar} ₹{monthly_avg[m]:,.0f}')

### 3.4 Correlation Analysis

In [ ]:
numeric_cols = ['unit_price','units_sold','discount_pct','revenue','profit_margin','profit','returns_pct','net_revenue']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10,8), facecolor='#0D1117')
ax.set_facecolor('#0D1117')
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.5, ax=ax, annot_kws={'size':9})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', color='#E6EDF3', pad=14)
plt.tight_layout(); plt.show()
print(f'Strongest correlation with net_revenue: {corr["net_revenue"].drop("net_revenue").abs().idxmax()}')

## 🔧 4. Feature Engineering

In [ ]:
df2 = df.copy()

# Encode categorical features
label_encoders = {}
for col in ['region','category','product','channel']:
    le = LabelEncoder()
    df2[col+'_enc'] = le.fit_transform(df2[col])
    label_encoders[col] = le

FEATURES = ['year','month','quarter','day_of_week',
            'unit_price','units_sold','discount_pct',
            'profit_margin','returns_pct',
            'region_enc','category_enc','channel_enc']
TARGET = 'net_revenue'

X = df2[FEATURES]
y = df2[TARGET]

print(f'Features: {len(FEATURES)}')
print(f'Target: {TARGET}')
print(f'X shape: {X.shape}')
X.describe().round(2)

## 🤖 5. Model Training & Evaluation

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest':     RandomForestRegressor(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=150, max_depth=5, learning_rate=0.08, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'MAE':  mean_absolute_error(y_test, preds),
        'RMSE': np.sqrt(mean_squared_error(y_test, preds)),
        'R2':   r2_score(y_test, preds),
        'model': model, 'preds': preds
    }
    print(f'{name:25s} → R²={results[name]["R2"]:.4f}  MAE=₹{results[name]["MAE"]:,.0f}  RMSE=₹{results[name]["RMSE"]:,.0f}')

In [ ]:
# Results comparison table
summary_df = pd.DataFrame({
    name: {'MAE (₹)': f'{r["MAE"]:,.0f}', 'RMSE (₹)': f'{r["RMSE"]:,.0f}', 'R² Score': f'{r["R2"]:.4f}'}
    for name, r in results.items()
}).T
print('\n=== Model Comparison ===')
print(summary_df)

In [ ]:
# Save best model
best_name = max(results, key=lambda k: results[k]['R2'])
best = results[best_name]
joblib.dump(best['model'], '../models/best_model.pkl')
print(f'\n🏆 Best Model: {best_name}')
print(f'   R² Score : {best["R2"]:.4f}')
print(f'   MAE      : ₹{best["MAE"]:,.0f}')
print(f'   RMSE     : ₹{best["RMSE"]:,.0f}')
print('\nModel saved → models/best_model.pkl ✅')

## 📈 6. Visualizations

In [ ]:
# Load pre-generated charts
from IPython.display import display, Image
import os

charts = [
    ('../outputs/chart5_model_comparison.png', 'Model Comparison'),
    ('../outputs/chart6_actual_vs_predicted.png', 'Actual vs Predicted'),
    ('../outputs/chart7_feature_importance.png', 'Feature Importance'),
]
for path, title in charts:
    if os.path.exists(path):
        print(f'\n--- {title} ---')
        display(Image(path))
    else:
        print(f'Run chart generation script first for: {title}')

## 🔮 7. Predict New Sales

In [ ]:
# Load saved model
loaded_model = joblib.load('../models/best_model.pkl')

# Example prediction
sample = pd.DataFrame([{
    'year': 2024, 'month': 11, 'quarter': 4, 'day_of_week': 4,
    'unit_price': 28000, 'units_sold': 12,
    'discount_pct': 0.10, 'profit_margin': 0.22, 'returns_pct': 0.03,
    'region_enc': 2,   # East
    'category_enc': 0, # Electronics
    'channel_enc': 0,  # Online
}])

predicted = loaded_model.predict(sample)[0]
print(f'🎯 Predicted Net Revenue: ₹{predicted:,.2f}')
print(f'   (~₹{predicted/1e3:.1f}K for an Electronics Online sale in Nov-East region)')

## ✅ 8. Summary & Key Insights

| Insight | Finding |
|---------|--------|
| **Best Model** | Gradient Boosting (R² = 0.9963) |
| **Top Feature** | `unit_price` × `units_sold` combination |
| **Peak Season** | October–December (Festive season, +40% revenue) |
| **Top Category** | Electronics (highest revenue, largest ticket size) |
| **Best Channel** | Wholesale (highest volume, deepest discounts) |
| **Best Region** | Balanced across regions; North slightly leads |

---

### 🚀 Future Improvements
- Add time-series forecasting (ARIMA / Prophet / LSTM)
- Hyperparameter tuning with GridSearchCV / Optuna
- Deploy as Flask/FastAPI REST endpoint
- Add real-time prediction dashboard with Streamlit
- Integrate with Power BI via Python visuals